<a href="https://colab.research.google.com/github/Vijayavallabh/ISRO-GeoNLI/blob/AMEP24/apps/pe/docs/pe_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Perception Encoder Demo
[![Paper](https://img.shields.io/badge/Paper-Perception%20Encoder-b31b1b.svg)](https://ai.meta.com/research/publications/perception-encoder-the-best-visual-embeddings-are-not-at-the-output-of-the-network)
[![Paper](https://img.shields.io/badge/arXiv-2504.13181-brightgreen.svg?style=flat-square)](https://arxiv.org/abs/2504.13181)
[![Hugging Face Collection](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Collection-blue)](https://huggingface.co/collections/facebook/perception-encoder-67f977c9a65ca5895a7f6ba1)
[![Model License](https://img.shields.io/badge/Model_License-Apache_2.0-olive)](https://opensource.org/licenses/Apache-2.0)

This notebook provides examples of image and video feature extraction with pre-trained Perception Encoder (PE). These featuire can be used for image and video zero-shot classification and retrieval.

You can run the demo locally or run it on Google colab. You can also run it with (faster) or wihtout GPU.

In [ ]:
# check whether run in Colab
if 'google.colab' in sys.modules:
    print('Running in Colab.')
    !git clone https://github.com/facebookresearch/perception_models.git
    !pip install decord
    !pip install ftfy
    sys.path.append('./perception_models')
    os.chdir('./perception_models')
else:
    sys.path.append('../../../')

import decord

if torch.cuda.is_available():
    print('GPU is available. Use GPU for this demo')
else:
    print('Use CPU for this demo')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import os, sys
import torch
import matplotlib.pyplot as plt
from PIL import Image
import core.vision_encoder.pe as pe
import core.vision_encoder.transforms as transforms


In [ ]:
print("PE configs:", pe.VisionTransformer.available_configs())
# PE configs: ['PE-Core-G14-448', 'PE-Core-L14-336', 'PE-Core-B16-224', 'PE-Core-S16-384', 'PE-Core-T16-384', 'PE-Lang-G14-448', 'PE-Lang-L14-448', 'PE-Lang-G14-448-Tiling', 'PE-Lang-L14-448-Tiling', 'PE-Spatial-G14-448', 'PE-Spatial-L14-448', 'PE-Spatial-B16-512', 'PE-Spatial-S16-512', 'PE-Spatial-T16-512']

model = pe.VisionTransformer.from_config("PE-Spatial-G14-448", pretrained=True)  # Loads from HF
model = model.cuda()

preprocess = transforms.get_image_transform(model.image_size)
image = preprocess(Image.open("docs/assets/cat.png")).unsqueeze(0).cuda()

out = model.forward_features(image, strip_cls_token=True)  # pass layer_idx=<idx> to get a specific layer's output!
print(out.shape)
# torch.Size([1, 1024, 1024])

Running in Colab.
Cloning into 'perception_models'...
remote: Enumerating objects: 747, done.
remote: Counting objects: 100% (239/239), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 747 (delta 191), reused 123 (delta 123), pack-reused 508 (from 2)
Receiving objects: 100% (747/747), 13.29 MiB | 20.97 MiB/s, done.
Resolving deltas: 100% (355/355), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
GPU is available. Use GPU for this demo
PE configs: ['PE-Core-G14-448', 'PE-Core-L14-336', 'PE-Core-B16-224', 'PE-Core-S16-384', 'PE-Core-T16-384', 'PE-Lang-G14-448', 'PE-Lang-L14-448', 'PE-Lang-G14-448-Tiling', 'PE-Lang-L14-448-Tiling', 'PE-Spatial-G14-448', 'PE-Spatial-L14-448', 'PE-Spatial-B16-512', 'PE-Spatial-S16-512', 'PE-Spatial-T16-512']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


PE-Spatial-G14-448.pt:   0%|          | 0.00/7.41G [00:00<?, ?B/s]

In [ ]:
import torch.nn as nn
class SpatialSegmentation(nn.Module):
    def __init__(self, pe_model, num_classes):
        super().__init__()
        self.encoder = pe_model
        feature_dim = pe_model.embed_dim

        # Decoder head
        self.decoder = nn.Sequential(
            nn.Conv2d(feature_dim, 256, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(256, num_classes, 1)
        )

    def forward(self, x):
        B = x.shape[0]
        # Get features: [B, N, D]
        features = self.encoder.forward_features(x, strip_cls_token=True)

        # Reshape to spatial grid
        # For 448x448 image with patch size 14: 32x32 grid
        h = w = int(features.shape[1] ** 0.5)
        features = features.transpose(1, 2).reshape(B, -1, h, w)

        # Decode to segmentation map
        seg_map = self.decoder(features)

        # Upsample to original resolution
        seg_map = torch.nn.functional.interpolate(
            seg_map, size=(448, 448), mode='bilinear', align_corners=False
        )
        return seg_map

# Usage
seg_model = SpatialSegmentation(model, num_classes=21).cuda()
seg_output = seg_model(image)  # [B, num_classes, H, W]

In [ ]:
"""
Text-Guided Segmentation Pipeline using PE-Spatial + CLIP/VLM + SAM

This pipeline:
1. Uses PE-Spatial to extract dense spatial features
2. Uses CLIP/VLM to find regions matching text description
3. Uses SAM to refine segmentation masks

Requirements:
pip install transformers segment-anything opencv-python torch torchvision
"""

import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
import cv2
from typing import List, Tuple, Optional
import matplotlib.pyplot as plt

# Assuming PE models are available
import core.vision_encoder.pe as pe
import core.vision_encoder.transforms as pe_transforms

# For VLM (using CLIP as example)
from transformers import CLIPProcessor, CLIPModel

# For SAM
try:
    from segment_anything import sam_model_registry, SamPredictor
except ImportError:
    print("Install SAM: pip install git+https://github.com/facebookresearch/segment-anything.git")


class TextGuidedSegmentation:
    """Pipeline for text-guided segmentation using PE-Spatial + VLM + SAM"""

    def __init__(
        self,
        pe_model_name: str = "PE-Spatial-G14-448",
        clip_model_name: str = "openai/clip-vit-large-patch14",
        sam_checkpoint: Optional[str] = None,
        sam_model_type: str = "vit_h",
        device: str = "cuda"
    ):
        self.device = device

        # 1. Load PE-Spatial encoder
        print("Loading PE-Spatial model...")
        self.pe_model = pe.VisionTransformer.from_config(pe_model_name, pretrained=True)
        self.pe_model = self.pe_model.to(device).eval()
        self.pe_transform = pe_transforms.get_image_transform(self.pe_model.image_size)

        # 2. Load CLIP for text-image matching
        print("Loading CLIP model...")
        self.clip_model = CLIPModel.from_pretrained(clip_model_name).to(device)
        self.clip_processor = CLIPProcessor.from_pretrained(clip_model_name)

        # 3. Load SAM for refinement
        if sam_checkpoint:
            print("Loading SAM model...")
            sam = sam_model_registry[sam_model_type](checkpoint=sam_checkpoint)
            sam = sam.to(device)
            self.sam_predictor = SamPredictor(sam)
        else:
            print("SAM checkpoint not provided. Skipping SAM refinement.")
            self.sam_predictor = None

    def extract_pe_features(self, image: Image.Image) -> Tuple[torch.Tensor, int, int]:
        """Extract dense spatial features from PE-Spatial model"""
        # Preprocess
        img_tensor = self.pe_transform(image).unsqueeze(0).to(self.device)

        # Extract features
        with torch.no_grad():
            features = self.pe_model.forward_features(img_tensor, strip_cls_token=True)
            # features shape: [1, num_patches, feature_dim]

        # Calculate spatial grid dimensions
        num_patches = features.shape[1]
        grid_size = int(np.sqrt(num_patches))

        return features, grid_size, grid_size

    def compute_text_similarity_map(
        self,
        image: Image.Image,
        text_queries: List[str],
        pe_features: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Compute similarity map between image regions and text queries

        Returns: similarity map of shape [H, W] with values 0-1
        """
        # Get PE features if not provided
        if pe_features is None:
            pe_features, h, w = self.extract_pe_features(image)
        else:
            num_patches = pe_features.shape[1]
            h = w = int(np.sqrt(num_patches))

        # Reshape PE features to spatial grid
        B, N, D = pe_features.shape
        pe_features_spatial = pe_features.reshape(B, h, w, D)

        # Process image patches for CLIP
        img_array = np.array(image.resize((self.pe_model.image_size, self.pe_model.image_size)))
        patch_size = self.pe_model.image_size // h

        # Extract patches
        patches = []
        for i in range(h):
            for j in range(w):
                patch = img_array[
                    i*patch_size:(i+1)*patch_size,
                    j*patch_size:(j+1)*patch_size
                ]
                patches.append(Image.fromarray(patch))

        # Encode patches with CLIP
        with torch.no_grad():
            # Encode all patches
            patch_inputs = self.clip_processor(images=patches, return_tensors="pt", padding=True)
            patch_inputs = {k: v.to(self.device) for k, v in patch_inputs.items()}
            patch_features = self.clip_model.get_image_features(**patch_inputs)
            patch_features = patch_features / patch_features.norm(dim=-1, keepdim=True)

            # Encode text queries
            text_inputs = self.clip_processor(text=text_queries, return_tensors="pt", padding=True)
            text_inputs = {k: v.to(self.device) for k, v in text_inputs.items()}
            text_features = self.clip_model.get_text_features(**text_inputs)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)

            # Compute similarities
            similarity = (patch_features @ text_features.T).max(dim=1)[0]  # Best match across queries
            similarity_map = similarity.reshape(h, w)

        return similarity_map

    def get_top_k_points(
        self,
        similarity_map: torch.Tensor,
        k: int = 5,
        threshold: float = 0.5
    ) -> np.ndarray:
        """Get top-k points from similarity map for SAM prompts"""
        similarity_np = similarity_map.cpu().numpy()

        # Threshold and get coordinates
        mask = similarity_np > threshold
        if mask.sum() == 0:
            # If no points above threshold, take top-k anyway
            flat_indices = np.argsort(similarity_np.flatten())[-k:]
        else:
            # Get top-k from thresholded regions
            masked_sim = np.where(mask, similarity_np, -np.inf)
            flat_indices = np.argsort(masked_sim.flatten())[-k:]

        h, w = similarity_np.shape
        points = np.array([[idx // w, idx % w] for idx in flat_indices])

        return points

    def refine_with_sam(
        self,
        image: Image.Image,
        similarity_map: torch.Tensor,
        num_points: int = 5,
        threshold: float = 0.5
    ) -> np.ndarray:
        """Use SAM to refine segmentation based on similarity map"""
        if self.sam_predictor is None:
            print("SAM not available. Returning upsampled similarity map.")
            sim_map = similarity_map.cpu().numpy()
            h, w = image.size[1], image.size[0]
            upsampled = cv2.resize(sim_map, (w, h), interpolation=cv2.INTER_LINEAR)
            return (upsampled > threshold).astype(np.uint8) * 255

        # Set image for SAM
        image_np = np.array(image)
        self.sam_predictor.set_image(image_np)

        # Get prompt points from similarity map
        grid_h, grid_w = similarity_map.shape
        img_h, img_w = image_np.shape[:2]

        # Get top-k points
        grid_points = self.get_top_k_points(similarity_map, k=num_points, threshold=threshold)

        # Scale points to image coordinates
        scale_h = img_h / grid_h
        scale_w = img_w / grid_w
        image_points = grid_points * np.array([scale_h, scale_w])
        image_points = image_points[:, [1, 0]]  # SAM expects (x, y)

        # Create labels (all positive prompts)
        point_labels = np.ones(len(image_points))

        # Predict mask with SAM
        masks, scores, logits = self.sam_predictor.predict(
            point_coords=image_points,
            point_labels=point_labels,
            multimask_output=True
        )

        # Return best mask
        best_mask_idx = np.argmax(scores)
        return (masks[best_mask_idx] * 255).astype(np.uint8)

    def segment(
        self,
        image: Image.Image,
        text_query: str,
        use_sam: bool = True,
        visualize: bool = True
    ) -> Tuple[np.ndarray, torch.Tensor]:
        """
        Main segmentation pipeline

        Args:
            image: Input PIL Image
            text_query: Text description of object to segment
            use_sam: Whether to use SAM for refinement
            visualize: Whether to show visualization

        Returns:
            mask: Binary segmentation mask (numpy array)
            similarity_map: Raw similarity map from VLM
        """
        print(f"Segmenting: '{text_query}'")

        # Step 1: Extract PE-Spatial features
        print("Extracting spatial features...")
        pe_features, h, w = self.extract_pe_features(image)

        # Step 2: Compute text-image similarity
        print("Computing text-image similarity...")
        similarity_map = self.compute_text_similarity_map(
            image, [text_query], pe_features
        )

        # Step 3: Refine with SAM
        if use_sam and self.sam_predictor is not None:
            print("Refining with SAM...")
            mask = self.refine_with_sam(image, similarity_map)
        else:
            # Simple thresholding
            sim_np = similarity_map.cpu().numpy()
            threshold = sim_np.mean() + 0.5 * sim_np.std()
            mask_low_res = (sim_np > threshold).astype(np.uint8) * 255
            mask = cv2.resize(mask_low_res, (image.width, image.height),
                            interpolation=cv2.INTER_NEAREST)

        # Visualization
        if visualize:
            self.visualize_results(image, similarity_map, mask, text_query)

        return mask, similarity_map

    def visualize_results(
        self,
        image: Image.Image,
        similarity_map: torch.Tensor,
        mask: np.ndarray,
        text_query: str
    ):
        """Visualize segmentation results"""
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # Original image
        axes[0].imshow(image)
        axes[0].set_title("Original Image")
        axes[0].axis('off')

        # Similarity map
        sim_np = similarity_map.cpu().numpy()
        im = axes[1].imshow(sim_np, cmap='jet')
        axes[1].set_title(f"Similarity Map\n'{text_query}'")
        axes[1].axis('off')
        plt.colorbar(im, ax=axes[1])

        # Final mask overlay
        axes[2].imshow(image)
        mask_overlay = np.zeros((*mask.shape, 4))
        mask_overlay[mask > 0] = [1, 0, 0, 0.5]  # Red with 50% transparency
        axes[2].imshow(mask_overlay)
        axes[2].set_title("Segmentation Result")
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()


# Example usage
if __name__ == "__main__":
    # Initialize pipeline
    # Note: Download SAM checkpoint from: https://github.com/facebookresearch/segment-anything#model-checkpoints
    sam_checkpoint = "path/to/sam_vit_h_4b8939.pth"  # Optional

    pipeline = TextGuidedSegmentation(
        pe_model_name="PE-Spatial-G14-448",
        clip_model_name="openai/clip-vit-large-patch14",
        sam_checkpoint=sam_checkpoint if os.path.exists(sam_checkpoint) else None,
        device="cuda" if torch.cuda.is_available() else "cpu"
    )

    # Load image
    image = Image.open("your_image.jpg").convert("RGB")

    # Segment with text query
    mask, similarity_map = pipeline.segment(
        image,
        text_query="a cat sitting on the floor",
        use_sam=True,
        visualize=True
    )

    # Save results
    Image.fromarray(mask).save("segmentation_mask.png")
    print(f"Mask saved! Shape: {mask.shape}")